In [5]:
import json
import os
import pandas as pd
from pathlib import Path
from hta.trace_diff import LabeledTrace, DeviceType, TraceDiff
from hta.utils.utils import shorten_name, flatten_column_names

In [9]:

shared_t = LabeledTrace(label="Shared", trace_dir=f"outputs/profile_trace_shared_layer/iteration_100")
baseline_t = LabeledTrace(label="Baseline", trace_dir=f"outputs/profile_trace_shared_layer_matched/iteration_100")

Parsed outputs/profile_trace_shared_layer/iteration_100/rank0_trace.json time = 0.10 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425
Parsed outputs/profile_trace_shared_layer/iteration_100/rank0_trace.json backend=ParserBackend.JSON in 0.25 seconds; current PID:2208035
Overall parsing of outputs/profile_trace_shared_layer/iteration_100/rank0_trace.json in 0.30 seconds; current PID:2208035
leaving parse_multiple_ranks duration=0.31 seconds
leaving parse_traces duration=0.31 seconds
Parsed outputs/profile_trace_shared_layer_matched/iteration_100/rank0_trace.json time = 0.05 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425
Parsed outputs/profile_trace_shared_layer_matched/iteration_100/rank0_trace.json backend=ParserBackend.JSON i

In [10]:


print(f"available ranks for shared: {shared_t.ranks()}")
print(f"available ranks for baseline: {baseline_t.ranks()}")
print(f"available iterations for shared: {shared_t.iterations()}")
print(f"available iterations for baseline: {baseline_t.iterations()}")



available ranks for shared: [0]
available ranks for baseline: [0]
available iterations for shared: [99]
available iterations for baseline: [99]


In [11]:
df_comp = TraceDiff.compare_traces(baseline_t, shared_t, 0, 0, 99, 99, DeviceType.GPU)
df_comp

,Baseline_counts,Baseline_total_duration,Shared_counts,Shared_total_duration,diff_counts,diff_duration,counts_change_categories
name,,,,,,,
Memcpy DtoD (Device -> Device),115.0,1095.0,160,1277.0,45.0,182.0,+
Memcpy DtoH (Device -> Pinned),4.0,6.0,4,7.0,0.0,1.0,=
Memcpy HtoD (Pageable -> Device),3.0,77.0,3,66.0,0.0,-11.0,=
Memset (Device),49.0,93.0,49,68.0,0.0,-25.0,=
ampere_bf16_s16816gemm_bf16_128x128_ldg8_f2f_stages_32x5_nn,84.0,33620.0,84,39460.0,0.0,5840.0,=
...,...,...,...,...,...,...,...
triton_red_fused__fused_rms_norm_backward__to_copy_add_mean_mul_pow_rsqrt_view_42,0.0,0.0,1,339.0,1.0,339.0,+
"void cublasLt::splitKreduce_kernel<32, 16, int, float, __nv_bfloat16, float, false, __nv_bfloat16, __nv_bfloat16, __nv_bfloat16, true, false, false>(cublasLt::cublasSplitKParams<float>, float const*, __nv_bfloat16 const*, __nv_bfloat16*, __nv_bfloat16*, float const*, float const*, __nv_bfloat16 const*, float const*, __nv_bfloat16*, void*, long, float*, int*, float*, float const*, float const*, float const*, float const*)",0.0,0.0,18,49.0,18.0,49.0,+
void cutlass::Kernel2<cutlass_80_tensorop_bf16_s16816gemm_relu_bf16_64x256_32x4_nn_align8>(cutlass_80_tensorop_bf16_s16816gemm_relu_bf16_64x256_32x4_nn_align8::Params),0.0,0.0,63,7397.0,63.0,7397.0,+


In [12]:
df_comp.sort_values(by="diff_duration",ascending=False)

,Baseline_counts,Baseline_total_duration,Shared_counts,Shared_total_duration,diff_counts,diff_duration,counts_change_categories
name,,,,,,,
ampere_bf16_s16816gemm_bf16_128x128_ldg8_f2f_stages_32x5_tn,109.0,102978.0,109,121967.0,0.0,18989.0,=
ampere_bf16_s16816gemm_bf16_256x128_ldg8_f2f_stages_64x3_nt,1.0,46922.0,13,65411.0,12.0,18489.0,+
ampere_bf16_s1688gemm_bf16_128x128_ldg8_f2f_stages_32x1_tn,0.0,0.0,94,13310.0,94.0,13310.0,+
triton_poi_fused__unsafe_view_add_fill_mul_sigmoid_silu_sub_view_10,0.0,0.0,5,10032.0,5.0,10032.0,+
void cutlass::Kernel2<cutlass_80_tensorop_bf16_s16816gemm_relu_bf16_64x256_32x4_tn_align8>(cutlass_80_tensorop_bf16_s16816gemm_relu_bf16_64x256_32x4_tn_align8::Params),0.0,0.0,94,9792.0,94.0,9792.0,+
...,...,...,...,...,...,...,...
triton_poi_fused__unsafe_view_mul_silu_10,12.0,6085.0,3,2024.0,-9.0,-4061.0,-
triton_per_fused__fused_rms_norm_backward__to_copy_add_mean_mul_pow_rsqrt_view_19,12.0,6063.0,2,1009.0,-10.0,-5054.0,-
triton_per_fused__fused_rms_norm_backward__to_copy__unsafe_view_add_mean_mul_pow_rsqrt_view_6,12.0,7387.0,2,1226.0,-10.0,-6161.0,-


In [6]:
shorten_name("void at::native::vectorized_elementwise_kernel<4, at::native::bfloat16_copy_kernel_cuda(at::TensorIteratorBase&)::{lambda(float)#1}, std::array<char*, 2ul> >(int, at::native::bfloat16_copy_kernel_cuda(at::TensorIteratorBase&)::{lambda(float)#1}, std::array<char*, 2ul>)	")

'at::native::vectorized_elementwise_kernel\t'

In [13]:
from hta.trace_analysis import TraceAnalysis

In [14]:
analyzer = TraceAnalysis(trace_dir="outputs/profile_trace_shared_layer/iteration_100")

Parsed outputs/profile_trace_shared_layer/iteration_100/rank0_trace.json time = 0.07 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425
Parsed outputs/profile_trace_shared_layer/iteration_100/rank0_trace.json backend=ParserBackend.JSON in 0.21 seconds; current PID:2208035
Overall parsing of outputs/profile_trace_shared_layer/iteration_100/rank0_trace.json in 0.26 seconds; current PID:2208035
leaving parse_multiple_ranks duration=0.27 seconds
leaving parse_traces duration=0.27 seconds
There is only one iteration in the trace. The analysis result may not be accurate.


In [15]:
cp_graph, success = analyzer.critical_path_analysis(
    rank = 0, annotation="", instance_id=0)

Trace does not contain CUDA Synchronization events so the results of analysis could be inaccurate.
Please see this PR to learn how to enable CUDA sync events https://github.com/pytorch/pytorch/pull/105187


In [16]:
cp_graph.summary()

Critical Path broken down by boundedness = (in % of duration)


bound_by
                               0.000000
cpu_bound                      0.149176
gpu_compute_bound             98.652322
gpu_kernel_kernel_overhead     1.195784
gpu_kernel_launch_overhead     0.002718
Name: duration, dtype: float64

['Memcpy DtoD (Device -> Device)',
 'Memcpy DtoH (Device -> Pinned)',
 'ampere_bf16_s16816gemm_bf16_128x128_ldg8_f2f_stages_32x5_nn',
 'ampere_bf16_s16816gemm_bf16_128x128_ldg8_f2f_stages_32x5_tn',
 'ampere_bf16_s16816gemm_bf16_128x128_ldg8_f2f_stages_64x3_nt',
 'ampere_bf16_s16816gemm_bf16_128x256_ldg8_f2f_stages_64x3_tn',
 'ampere_bf16_s16816gemm_bf16_256x128_ldg8_f2f_stages_64x3_nn',
 'ampere_bf16_s16816gemm_bf16_256x128_ldg8_f2f_stages_64x3_nt',
 'triton_per_fused__to_copy__unsafe_view_add_mean_mul_pow_rsqrt_8',
 'triton_poi_fused__scaled_dot_product_flash_attention__to_copy__unsafe_view_clone_expand_transpose_unsqueeze_view_5',
 'triton_poi_fused__to_copy_1',
 'triton_poi_fused__to_copy_2',
 'triton_red_fused__log_softmax__log_softmax_backward_data__to_copy_nll_loss_backward_nll_loss_forward_view_0',
 'triton_red_fused__log_softmax__to_copy_prepare_softmax_online_view_0',
 'void at::native::(anonymous namespace)::CatArrayBatchedCopy_alignedK_contig<at::native::(anonymous namespace

In [26]:
df_comp.columns

Index(['Baseline_counts', 'Baseline_total_duration', 'Shared_counts',
       'Shared_total_duration', 'diff_counts', 'diff_duration',
       'counts_change_categories'],
      dtype='object')

In [36]:
worse_ops = cp_graph.get_critical_path_breakdown()["s_name"].isin(df_comp.loc[df_comp["diff_duration"] > 0].index.tolist())

In [38]:
cp_graph.get_critical_path_breakdown().loc[worse_ops].sort_values(by="duration",ascending=False).head(10)

,event_idx,duration,type,s_name,cat,pid,tid,stream,index,bound_by
4583,9998.0,73397.0,critical_path_operator,ampere_bf16_s16816gemm_bf16_128x128_ldg8_f2f_s...,234.0,0,7,7.0,9998.0,gpu_compute_bound
3892,10050.0,58938.0,critical_path_operator,ampere_bf16_s16816gemm_bf16_256x128_ldg8_f2f_s...,234.0,0,7,7.0,10050.0,gpu_compute_bound
716,10040.0,55497.0,critical_path_operator,ampere_bf16_s16816gemm_bf16_256x128_ldg8_f2f_s...,234.0,0,7,7.0,10040.0,gpu_compute_bound
4387,10030.0,38481.0,critical_path_operator,triton_red_fused__log_softmax__log_softmax_bac...,234.0,0,7,7.0,10030.0,gpu_compute_bound
4140,10006.0,27504.0,critical_path_operator,triton_red_fused__log_softmax__to_copy_prepare...,234.0,0,7,7.0,10006.0,gpu_compute_bound
1261,17685.0,2404.0,critical_path_kernel_kernel_delay,Memcpy DtoD (Device -> Device),27.0,0,7,7.0,17685.0,gpu_kernel_kernel_overhead
4034,14948.0,2014.0,critical_path_operator,triton_poi_fused__unsafe_view_add_fill_mul_sig...,234.0,0,7,7.0,14948.0,gpu_compute_bound
1632,13652.0,2010.0,critical_path_operator,triton_poi_fused__unsafe_view_add_fill_mul_sig...,234.0,0,7,7.0,13652.0,gpu_compute_bound
3907,12438.0,2008.0,critical_path_operator,triton_poi_fused__unsafe_view_add_fill_mul_sig...,234.0,0,7,7.0,12438.0,gpu_compute_bound
324,10738.0,2000.0,critical_path_operator,triton_poi_fused__unsafe_view_add_fill_mul_sig...,234.0,0,7,7.0,10738.0,gpu_compute_bound


In [31]:
cp_graph.columns

AttributeError: 'CPGraph' object has no attribute 'columns'

In [19]:
cp_graph.get_critical_path_breakdown().sort_values(by="duration",ascending=False).head(10)

,event_idx,duration,type,s_name,cat,pid,tid,stream,index,bound_by
4583,9998.0,73397.0,critical_path_operator,ampere_bf16_s16816gemm_bf16_128x128_ldg8_f2f_s...,234.0,0,7,7.0,9998.0,gpu_compute_bound
3892,10050.0,58938.0,critical_path_operator,ampere_bf16_s16816gemm_bf16_256x128_ldg8_f2f_s...,234.0,0,7,7.0,10050.0,gpu_compute_bound
716,10040.0,55497.0,critical_path_operator,ampere_bf16_s16816gemm_bf16_256x128_ldg8_f2f_s...,234.0,0,7,7.0,10040.0,gpu_compute_bound
4387,10030.0,38481.0,critical_path_operator,triton_red_fused__log_softmax__log_softmax_bac...,234.0,0,7,7.0,10030.0,gpu_compute_bound
4140,10006.0,27504.0,critical_path_operator,triton_red_fused__log_softmax__to_copy_prepare...,234.0,0,7,7.0,10006.0,gpu_compute_bound
3786,10302.0,18689.0,critical_path_operator,pytorch_flash::flash_bwd_dq_dk_dv_loop_seqk_pa...,234.0,0,7,7.0,10302.0,gpu_compute_bound
1250,11040.0,17194.0,critical_path_operator,pytorch_flash::flash_bwd_dq_dk_dv_loop_seqk_pa...,234.0,0,7,7.0,11040.0,gpu_compute_bound
1624,11602.0,17014.0,critical_path_operator,pytorch_flash::flash_bwd_dq_dk_dv_loop_seqk_pa...,234.0,0,7,7.0,11602.0,gpu_compute_bound
157,16267.0,16994.0,critical_path_operator,pytorch_flash::flash_bwd_dq_dk_dv_loop_seqk_pa...,234.0,0,7,7.0,16267.0,gpu_compute_bound
195,12728.0,16971.0,critical_path_operator,pytorch_flash::flash_bwd_dq_dk_dv_loop_seqk_pa...,234.0,0,7,7.0,12728.0,gpu_compute_bound
